## P35
- Let me kinda sketch out here what expanding these input images into patches will look like
- Then maybe we animate the breaking up/expansion with manim, we'll see. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image
import copy
import math
import time

import torch
import torch.nn.functional as F

# openpi imports
from openpi.training import config as _config
from openpi.policies import policy_config as _policy_config
from openpi.policies import aloha_policy
from openpi.shared import download
from openpi import transforms as _transforms
from openpi.shared import normalize as _normalize
from openpi.models import model as _model

print("Imports OK")

In [ ]:
joint_labels = [
    "L joint 0 (waist)", "L joint 1 (shoulder)", "L joint 2 (elbow)",
    "L joint 3 (forearm)", "L joint 4 (wrist angle)", "L joint 5 (wrist rotate)",
    "L gripper",
    "R joint 0 (waist)", "R joint 1 (shoulder)", "R joint 2 (elbow)",
    "R joint 3 (forearm)", "R joint 4 (wrist angle)", "R joint 5 (wrist rotate)",
    "R gripper",
]

In [ ]:
def frame_to_obs(frame):
    """Convert a HuggingFace dataset frame to the observation dict
    that the policy expects (post-RepackTransform format)."""
    def to_chw(img):
        arr = np.array(img)  # PIL -> HWC (H, W, 3)
        return arr.transpose(2, 0, 1)  # -> CHW (3, H, W)
    
    return {
        "images": {
            "cam_high": to_chw(frame["observation.images.cam_high"]),
            "cam_left_wrist": to_chw(frame["observation.images.cam_left_wrist"]),
            "cam_right_wrist": to_chw(frame["observation.images.cam_right_wrist"]),
        },
        "state": np.array(frame["observation.state"]),
        "prompt": "uncap the pen",
    }

In [ ]:
ds = load_dataset("physical-intelligence/aloha_pen_uncap_diverse", split="train")
print(f"Total rows: {len(ds)}")
print(f"Columns: {ds.column_names}")
print(f"Episode indices: {sorted(set(ds['episode_index']))}")

In [ ]:
# Pick an episode and grab all its frames
EPISODE_IDX = 0

ep_indices = np.array(ds["episode_index"])
mask = ep_indices == EPISODE_IDX
row_indices = np.where(mask)[0]
episode = ds.select(row_indices)

In [ ]:
# Pick a frame in the middle of the episode
FRAME_IDX = 150
frame = episode[FRAME_IDX]

In [ ]:
# Display camera views
cam_names = ["cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, cam_names):
    img = frame[f"observation.images.{name}"]
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.suptitle(f"Episode {EPISODE_IDX}, Frame {FRAME_IDX} (t={frame['timestamp']:.2f}s)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
config = _config.get_config("pi0_aloha_pen_uncap")

# Point to converted PyTorch checkpoint
# checkpoint_dir = "../data/robots/openpi/pi0_aloha_pen_uncap_pytorch"
checkpoint_dir = "../data/openpi/pi0_aloha_pen_uncap_pytorch"

print(f"Model type: {config.model.model_type}")
print(f"Action dim: {config.model.action_dim}")
print(f"Action horizon: {config.model.action_horizon}")

In [ ]:
# Create the trained policy — auto-detects PyTorch format
policy = _policy_config.create_trained_policy(config, checkpoint_dir)
print(f"Policy created successfully!")
print(f"PyTorch model: {policy._is_pytorch_model}")
print(f"Device: {policy._pytorch_device}")
print(f"Number of input transforms: {len(policy._input_transform.transforms)}")
print(f"Number of output transforms: {len(policy._output_transform.transforms)}")

In [ ]:
# Quick reference inference to make sure the model works
obs_test = frame_to_obs(frame)
result_test = policy.infer(obs_test)
print(f"Actions shape: {result_test['actions'].shape}")
print(f"Inference time: {result_test['policy_timing']['infer_ms']:.1f} ms")

In [ ]:
# Fix noise for reproducibility (numpy, no JAX dependency)
rng = np.random.RandomState(42)
fixed_noise = rng.randn(50, 32).astype(np.float32)
print(f"fixed_noise: {fixed_noise.shape}, dtype={fixed_noise.dtype}")
print(f"  norm={np.linalg.norm(fixed_noise):.4f}, mean={fixed_noise.mean():.4f}")

# Reference result using policy.infer as a black box
obs_ref = frame_to_obs(frame)
result_ref = policy.infer(obs_ref, noise=fixed_noise)
print(f"\nReference actions: {result_ref['actions'].shape}")
print(f"  range: [{result_ref['actions'].min():.4f}, {result_ref['actions'].max():.4f}]")

In [ ]:
device = torch.device(policy._pytorch_device)
pt_model = policy._model  # the PI0Pytorch instance
print(f"Device: {device}")
print(f"Model type: {type(pt_model).__name__}")

In [ ]:
# Step 1: Input transforms (same chain as JAX version)
obs_manual = frame_to_obs(frame)
inputs = policy._input_transform(obs_manual)

print("After input transforms:")
for k, v in inputs.items():
    if isinstance(v, dict):
        for k2, v2 in v.items():
            print(f"  {k}/{k2}: {v2.shape}, dtype={v2.dtype}")
    elif isinstance(v, np.ndarray):
        print(f"  {k}: {v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {type(v)}")

In [ ]:
# Step 2: Batch and convert to PyTorch tensors — each leaf [shape] -> torch [1, shape]
import jax  # just for tree.map utility, no JAX computation
inputs_pt = jax.tree.map(
    lambda x: torch.from_numpy(np.array(x)).to(device)[None, ...],
    inputs
)

print("Batched PyTorch tensors:")
for k, v in inputs_pt.items():
    if isinstance(v, dict):
        for k2, v2 in v.items():
            print(f"  {k}/{k2}: {v2.shape}, dtype={v2.dtype}, device={v2.device}")
    elif isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}, dtype={v.dtype}")

In [ ]:
# Step 3: Build Observation dataclass
# For torch uint8 images, from_dict converts to float32 and permutes to BCHW
observation = _model.Observation.from_dict(inputs_pt)

print("Observation:")
for name, img in observation.images.items():
    print(f"  image[{name}]: {img.shape}, dtype={img.dtype}, range=[{img.min():.2f}, {img.max():.2f}]")
for name, m in observation.image_masks.items():
    print(f"  mask[{name}]: {m.shape}, dtype={m.dtype}")
print(f"  state: {observation.state.shape}, dtype={observation.state.dtype}")
print(f"  tokenized_prompt: {observation.tokenized_prompt.shape}")
print(f"  tokenized_prompt_mask: {observation.tokenized_prompt_mask.shape}, valid={observation.tokenized_prompt_mask.sum()}")

In [ ]:
# Step 4: Preprocess observation (no augmentation at inference)
# This unpacks the Observation into lists that embed_prefix expects
images, img_masks, lang_tokens, lang_masks, state = pt_model._preprocess_observation(
    observation, train=False
)

print("Preprocessed observation:")
for i, (img, mask) in enumerate(zip(images, img_masks)):
    print(f"  image[{i}]: {img.shape}, dtype={img.dtype}, range=[{img.min():.2f}, {img.max():.2f}]")
    print(f"  mask[{i}]:  {mask.shape}, val={mask}")
print(f"  lang_tokens: {lang_tokens.shape}, dtype={lang_tokens.dtype}")
print(f"  lang_masks: {lang_masks.shape}, valid={lang_masks.sum()}")
print(f"  state: {state.shape}, dtype={state.dtype}")

In [ ]:
# Step 5: Embed prefix using model's embed_prefix method
with torch.no_grad():
    prefix_embs, prefix_pad_masks, prefix_att_masks = pt_model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks
    )

print(f"prefix_embs:      {prefix_embs.shape}, dtype={prefix_embs.dtype}")      # [1, 816, 2048]
print(f"prefix_pad_masks: {prefix_pad_masks.shape}, dtype={prefix_pad_masks.dtype}")  # [1, 816]
print(f"prefix_att_masks: {prefix_att_masks.shape}, dtype={prefix_att_masks.dtype}")  # [1, 816]

In [ ]:
prefix_embs.shape

In [ ]:
plt.imshow(prefix_embs[0, :768, :].cpu().float().numpy())

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")

# Decode the full sequence
print(tokenizer.decode(lang_tokens[0]))

# See individual tokens (including padding)
tokens = tokenizer.convert_ids_to_tokens(lang_tokens[0].tolist())
for i, (tok, mask) in enumerate(zip(tokens, lang_masks[0])):
    print(f"  [{i:2d}] id={lang_tokens[0, i]:6d}  mask={mask.item()}  '{tok}'")

## Write ful size images to disk

In [ ]:
import os
from tqdm import tqdm

In [ ]:
# save_dir='../data/robots/p35'

# os.makedirs(save_dir, exist_ok=True)

In [ ]:
# inputs['image'].keys()

In [ ]:
# # Pick an episode and grab all its frames
# EPISODE_IDX = 0

# ep_indices = np.array(ds["episode_index"])
# mask = ep_indices == EPISODE_IDX
# row_indices = np.where(mask)[0]
# episode = ds.select(row_indices)

# # Pick a frame in the middle of the episode
# for image_name in ['base_0_rgb', 'left_wrist_0_rgb', 'right_wrist_0_rgb']:
#     im_dir = save_dir + '/full_size_' + image_name
#     os.makedirs(im_dir, exist_ok=True)
#     for FRAME_IDX in tqdm(range(300)):
   
#         frame = episode[FRAME_IDX]
        
#         obs_manual = frame_to_obs(frame)
#         inputs = policy._input_transform(obs_manual)
        
#         img_array=inputs['image'][image_name]
#         img=Image.fromarray(img_array)

#         img.save(im_dir+'/'+str(FRAME_IDX).zfill(3)+'.jpg')

### Write a bunch of patches to disk

In [ ]:
# # Pick an episode and grab all its frames
# EPISODE_IDX = 0

# ep_indices = np.array(ds["episode_index"])
# mask = ep_indices == EPISODE_IDX
# row_indices = np.where(mask)[0]
# episode = ds.select(row_indices)

# # Pick a frame in the middle of the episode
# for FRAME_IDX in tqdm(range(300)):
#     for image_name in ['base_0_rgb', 'left_wrist_0_rgb', 'right_wrist_0_rgb']:
#     # image_name='base_0_rgb' # 'left_wrist_0_rgb', 'right_wrist_0_rgb'

#         frame = episode[FRAME_IDX]
        
#         obs_manual = frame_to_obs(frame)
#         inputs = policy._input_transform(obs_manual)
        
#         img_array=inputs['image'][image_name]
#         img=Image.fromarray(img_array)
        
#         height, width = img_array.shape[:2]
        
#         grid_n = 16
#         patch_h = height // grid_n
#         patch_w = width // grid_n
#         total_height = 3.0
#         patch_size = total_height / grid_n
        
#         patch_dir = save_dir + '/' + str(FRAME_IDX)+'/'+image_name
#         os.makedirs(patch_dir, exist_ok=True)
        
#         for i in range(grid_n):
#             for j in range(grid_n):
#                 patch_img = img.crop((j*patch_w, i*patch_h, (j+1)*patch_w, (i+1)*patch_h))
#                 patch_path = os.path.join(patch_dir, f'patch_{i}_{j}.png')
#                 patch_img.save(patch_path)

In [ ]:
def split_with_padding(img, padding=4, skip_rows=2):
    N = img.shape[0]
    p = N // 16
    rows = range(skip_rows, 16 - skip_rows)
    h = len(rows) * p + padding * (len(rows) - 1)
    w = N + padding * 15
    out = np.zeros((h, w, 4), dtype=np.uint8)  # RGBA, alpha=0 everywhere
    for oi, i in enumerate(rows):
        for j in range(16):
            sy = oi * (p + padding)
            sx = j * (p + padding)
            out[sy:sy+p, sx:sx+p, :3] = img[i*p:(i+1)*p, j*p:(j+1)*p]
            out[sy:sy+p, sx:sx+p, 3] = 255  # opaque where there's a patch
    return out

im_split = split_with_padding(inputs['image']['right_wrist_0_rgb'], padding=2)
Image.fromarray(im_split, mode='RGBA').save('../data/ai_book_vol_2/3_vla/graphics/split_im_right_3.png')

In [ ]:
im_split=split_with_padding(inputs['image']['base_0_rgb'], padding=2)
Image.fromarray(im_split, mode='RGBA').save('../data/ai_book_vol_2/3_vla/graphics/split_im_base_3.png')
# plt.imshow(im_split)

In [ ]:
im_split=split_with_padding(inputs['image']['left_wrist_0_rgb'], padding=2)
Image.fromarray(im_split,  mode='RGBA').save('../data/ai_book_vol_2/3_vla/graphics/split_im_left_3.png')
plt.imshow(im_split)

In [ ]:
inputs['image'].keys()

In [ ]:
plt.imshow(inputs['image']['base_0_rgb'])

In [ ]:
plt.imshow(inputs['image']['left_wrist_0_rgb'])